In [ ]:
!pip install -q "transformers>=4.48.0" "peft>=0.12" "trl>=0.9" "bitsandbytes>=0.43" "datasets" "accelerate" "scikit-learn" "pandas" "tqdm"

import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
dataset = load_dataset("lmsys/lmsys-arena-human-preference-55k", split="train")
df = dataset.to_pandas()

# Create label: 0 = A wins, 1 = B wins, 2 = Tie
def get_label(row):
    if row.get("winner_model_a", 0) == 1: return 0
    if row.get("winner_model_b", 0) == 1: return 1
    return 2

df["label"] = df.apply(get_label, axis=1)

# Clean and truncate
df = df.dropna(subset=["prompt", "response_a", "response_b"])
MAX_CHARS = 1200
df["prompt"] = df["prompt"].str[:MAX_CHARS]
df["response_a"] = df["response_a"].str[:MAX_CHARS]
df["response_b"] = df["response_b"].str[:MAX_CHARS]

# Split: 10,000 train | 2,000 validation | 2,000 test
train_pool, temp_df = train_test_split(df, test_size=4000, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=2000, stratify=temp_df["label"], random_state=42)
train_df, _ = train_test_split(train_pool, train_size=10000, stratify=train_pool["label"], random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

README.md:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B /  184MB            

train.csv: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Train: 10000, Val: 2000, Test: 2000


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "microsoft/Phi-3.5-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="bfloat16",
    device_map="auto",
    trust_remote_code=False
)
model.eval()
print("Model loaded successfully.")

config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

... (5 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
import gc
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from transformers import BitsAndBytesConfig
from datasets import Dataset

gc.collect()
torch.cuda.empty_cache()

# 1. Format Data for SFT (Explicit prompt/completion)
def format_for_sft(row):
    prompt_text = f"Compare these two responses and choose the better one.\n\nPrompt: {row['prompt']}\n\nResponse A: {row['response_a']}\n\nResponse B: {row['response_b']}"
    answer = ["A", "B", "Tie"][row["label"]]
    return {
        "prompt": prompt_text,
        "completion": answer
    }

# Limit to 2000 examples
train_subset = train_df.head(2000)
train_sft = Dataset.from_pandas(train_subset[["prompt", "response_a", "response_b", "label"]]).map(
    format_for_sft, remove_columns=["prompt", "response_a", "response_b", "label"]
)
val_sft = Dataset.from_pandas(val_df[["prompt", "response_a", "response_b", "label"]]).map(
    format_for_sft, remove_columns=["prompt", "response_a", "response_b", "label"]
)

# 2. 4-bit Quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
)

model_ft = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=False
)
model_ft = prepare_model_for_kbit_training(model_ft)
model_ft.gradient_checkpointing_enable()

# 3. Training Setup
peft_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

trainer = SFTTrainer(
    model=model_ft,
    args=SFTConfig(
        output_dir="./lora_sft",
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="epoch",
        max_length=512,
        packing=False,
        report_to="none",
        fp16=True,
        bf16=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        completion_only_loss=True  # Now this works correctly with explicit prompt/completion
    ),
    train_dataset=train_sft,
    eval_dataset=val_sft,
    peft_config=peft_config,
)

# 4. T4 Fix for BFloat16 error
for param in trainer.model.parameters():
    if param.requires_grad and param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)

# 5. Train & Save
trainer.train()
trainer.save_model("./lora_sft/final")
print("Condition B Training Complete.")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

... (5 outputs left out)

Truncating eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Step,Training Loss
10,11.752838
20,4.235706
30,1.892538
40,0.736375
50,1.404305
60,0.684889
70,0.709583
80,0.542987
90,0.686032
100,0.719845


Condition B Training Complete.


In [ ]:
import gc
import torch

try:
    del model
    del tokenizer
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

GPU memory free: 10.36 GB


In [ ]:
from transformers import AutoTokenizer
model_id = "microsoft/Phi-3.5-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer reloaded successfully.")

Tokenizer reloaded successfully.


In [ ]:
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import pandas as pd

# Load the fine-tuned model
base_model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=False
)
model_B = PeftModel.from_pretrained(base_model, "./lora_sft/final")
model_B.eval()

def predict_finetuned(prompt, response_a, response_b, model_obj):
    instruction = f"Compare these two responses and choose the better one.\n\nPrompt: {prompt}\n\nResponse A: {response_a}\n\nResponse B: {response_b}"
    messages = [{"role": "user", "content": instruction}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model_obj.device)

    with torch.no_grad():
        outputs = model_obj.generate(
            **inputs, max_new_tokens=3, do_sample=False, pad_token_id=tokenizer.eos_token_id
        )

    prompt_len = inputs["input_ids"].shape[-1]
    text = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip().upper()

    # Smart Parser
    if text.startswith("A") or "RESPONSE A" in text: return 0
    elif text.startswith("B") or "RESPONSE B" in text: return 1
    return 2

print("Evaluating Condition B on 2,000 test examples...")
preds_B, truths_B = [], []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    preds_B.append(predict_finetuned(row["prompt"], row["response_a"], row["response_b"], model_B))
    truths_B.append(row["label"])

acc_B = accuracy_score(truths_B, preds_B)
f1_B = f1_score(truths_B, preds_B, average="macro")

print("\n" + "="*50)
print("CONDITION B (LoRA 2,000 Examples) RESULTS")
print("="*50)
print(f"Accuracy: {acc_B:.4f}")
print(f"Macro F1: {f1_B:.4f}")
print("="*50)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Evaluating Condition B on 2,000 test examples...


100%|██████████| 2000/2000 [18:55<00:00,  1.76it/s]


CONDITION B (LoRA 2,000 Examples) RESULTS
Accuracy: 0.3785
Macro F1: 0.3019


In [ ]:
!pip freeze > requirements.txt